<a href="https://colab.research.google.com/github/Dimildizio/python_course/blob/main/basics/oop_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ООП в Python

---
## Тема 1 — `__init__` и `self`

Ты уже умеешь хранить данные в словаре

In [ ]:
# Словарь отлично справляется с простым хранением данных
marine = {"name": "Titus", "chapter": "Ultramarines", "wounds": 2}
print(marine["name"])

Но когда появляетю

### Что это и зачем

Ты уже умеешь писать функции. Класс — это как функция, но с **памятью**.

```python
def recruit(name):   # обычная функция — не помнит данные между вызовами
    print(f'Боец {name} прибыл')
```

Но если нам нужен полноценный **Space Marine** — с именем, орденом и силовой бронёй — данные должны **жить внутри объекта** и быть доступны из любого метода.

**`__init__`** — это ритуал инициации. Python вызывает его **автоматически** при создании объекта. Именно здесь задаём все поля — и боец сразу готов к бою.

**`self`** — это сам объект, ссылка на конкретного бойца. Через `self.имя` ты прикрепляешь данные к этому экземпляру.

> **Правило:** если данные нужны в нескольких методах — они должны быть в `__init__` через `self`.


### Пример


In [ ]:
class SpaceMarine:
    def __init__(self, name, chapter):
        self.name = name
        self.chapter = chapter
        self.wounds = 2

    def battle_cry(self):
        if self.chapter == 'Ultramarines':
            print(f'{self.name} кричит: Макрагг превыше всего!')
        elif self.chapter == 'Space Wolves':
            print(f'{self.name} кричит: Людмила Ракка!')
        else:
            print(f'{self.name} кричит: За Императора!')

titus = SpaceMarine('Titus', 'Ultramarines')
ragnar = SpaceMarine('Ragnar Blackmane', 'Space Wolves')

titus.battle_cry()
ragnar.battle_cry()
print(titus.__dict__)


**Сравни: поле через метод vs через `__init__`**


In [ ]:
# ❌ Хрупко: объект не готов сразу после создания
class MarineBad:
    def set_name(self, name):
        self.name = name

    def battle_cry(self):
        print(f'{self.name} кричит: За Императора!')

marine = MarineBad()
# marine.battle_cry()  # раскомментируй — получишь AttributeError
marine.set_name('Titus')
marine.battle_cry()


Можно использовать @dataclass
там init уже есть автоматом

In [ ]:
from dataclasses import dataclass

@dataclass
class SpaceMarine:
    name: str
    chapter: str
    wounds: int = 2

### 📝 Задание 1


In [ ]:
# Напиши класс Ork с полями name и clan.
# Добавь метод waaagh() который печатает: 'Орк Grimskull из клана Goffs кричит: WAAAGH!'
# Создай двух орков из разных кланов, вызови waaagh() у каждого.

# твой код здесь


In [ ]:
# Проверь:
# ork = Ork('Grimskull', 'Goffs')
# print(ork.__dict__)  # {'name': 'Grimskull', 'clan': 'Goffs'}


### 🔍 Примечание

**`self` — это просто договорённость, не ключевое слово**

Python не требует слова `self` — это конвенция, которую все соблюдают чтобы код был читаемым. Технически можно написать `this`, `me`, `marine` или вообще `x` — и всё будет работать:

```python
class SpaceMarine:
    def __init__(marine, name):   # работает, но не делай так
        marine.name = name

    def battle_cry(this):         # тоже работает, но все будут в шоке
        print(this.name)
```

Просто всегда пиши `self` — это стандарт индустрии.

---

**Когда нормально назначать поля вне `__init__`?**

Иногда поле объективно не может существовать при создании — и это окей:

```python
class SpaceMarine:
    def __init__(self, name):
        self.name = name
        self.squad = None      # ок: поле есть, просто пустое

    def assign_squad(self, squad):
        self.squad = squad     # назначается позже — логично
```

Разница между **плохим** и **нормальным** назначением вне `__init__`:

| Ситуация | Оценка |
|---|---|
| Поле появляется только после вызова метода, нет `None` в `__init__` | ❌ хрупко |
| Поле объявлено в `__init__` как `None`, метод его заполняет позже | ✅ нормально |
| Поле появляется по результату внешнего события (загрузка из БД, API) | ✅ нормально |

**Паттерн Builder vs `__init__`:**

`__init__` создаёт объект сразу целиком. Подходит когда все данные известны заранее.

Builder нужен когда объект собирается **поэтапно** или конфигурация очень сложная:

```python
# __init__ — всё сразу, просто
marine = SpaceMarine('Titus', 'Ultramarines', weapon='Bolter')

# Builder — когда параметров 10+ и большинство опциональны
marine = (MarineBuilder('Titus')
    .chapter('Ultramarines')
    .weapon('Bolter')
    .with_terminator_armour()
    .build())
```

В Python Builder используется редко — `__init__` с дефолтными аргументами обычно справляется.


---
## Тема 2 — Поля и методы

### Что это и зачем

**Поля** — данные объекта. Хранятся через `self.имя`. (Про keywords и опциональные аргументы ты уже знаешь)
**Методы** — действия объекта. Читают и меняют поля через `self`.

- Поля: **«что мы знаем об объекте?»** — имя, очки жизни, оружие
- Методы: **«что объект умеет делать?»** — атаковать, получать урон

Методы всегда принимают `self` первым. Когда пишешь `marine.attack(enemy)`, Python сам подставляет `marine` как `self`.


### Пример


In [ ]:
class Warrior:
    def __init__(self, name, wounds, strength):
        self.name = name
        self.wounds = wounds
        self.strength = strength
        self.alive = True

    def attack(self, target):
        print(f'{self.name} атакует {target.name} с силой {self.strength}!')
        target.take_damage(self.strength)

    def take_damage(self, damage):
        self.wounds -= damage
        if self.wounds <= 0:
            self.alive = False
            print(f'{self.name} пал в бою. Слава Императору!')
        else:
            print(f'{self.name} получил {damage} урона. Осталось ран: {self.wounds}')

    def status(self):
        state = 'жив' if self.alive else 'мёртв'
        print(f'{self.name}: раны={self.wounds}, статус={state}')


marine = Warrior('Brother Titus', 4, 3)
heretic = Warrior('Chaos Cultist', 1, 1)

marine.status()
marine.attack(heretic)
heretic.attack(marine)
marine.status()


### 📝 Задание 2


In [ ]:
# Напиши класс Bolter (болтер — оружие Space Marines).
# Поля: name, shots (текущий боезапас), max_shots (максимум)
#
# Методы:
#   fire(n)   — стреляет n раз. Если патронов не хватает — предупреждение
#   reload()  — восстанавливает shots до max_shots
#   status()  — печатает: 'Boltgun: 24/30 патронов'

# твой код здесь


In [ ]:
# Проверь:
# gun = Bolter('Boltgun', 30)
# gun.fire(10)
# gun.status()   # Boltgun: 20/30 патронов
# gun.fire(25)   # недостаточно патронов
# gun.reload()
# gun.status()   # Boltgun: 30/30 патронов


### 🔍 Примечание

**Кроме обычных методов существуют ещё три вида — просто знай что они есть (их мы изучим потом):**

**Классовые методы (`@classmethod`)** — работают с классом, а не с конкретным объектом, когда не нужны объекты конкретного инстанса. Используются как альтернативные конструкторы:
```python
@classmethod
def from_roster(cls, roster_string):  # 'Titus:Ultramarines'
    name, chapter = roster_string.split(':')
    return cls(name, chapter)  # создаёт объект
```

**Статические методы (`@staticmethod`)** — вообще не знают ни про объект, ни про класс. По сути обычная функция, но логически относится к классу и только поэтому она в нем:
```python
@staticmethod
def roll_dice(sides=6):
    import random
    return random.randint(1, sides)
```

**Приватные методы** — имя начинается с `_` или `__`. Соглашение: "этот метод только для внутреннего использования, не трогай снаружи" - тоже по сути просто конвенция, даже два прочерка обходятся легко:
```python
def _calculate_wounds(self):   # один прочерк — "лучше не трогать"
    ...
def __secret_ritual(self):     # два прочерка — name mangling, реально скрыт
    ...
```

**Магические методы (`__dunder__`)** — Python вызывает их автоматически в определённых ситуациях. `__init__` — один из них. Другие примеры:
```python
def __str__(self):   # вызывается при print(obj)
    return f'Space Marine: {self.name}'

def __len__(self):   # вызывается при len(obj)
    return self.wounds

def __eq__(self, other):  # вызывается при obj1 == obj2
    return self.name == other.name

len(marine)
```


---
## Тема 3 — Наследование

### Что это и зачем

Наследование — один класс **берёт всё** из другого и добавляет своё.

В Warhammer 40K все юниты имеют общее: имя, раны, броню. Но Space Marine и Tyranid ведут себя по-разному.

Вместо того чтобы писать одно и то же дважды — создаём базовый `Unit`, а дочерние классы наследуют его и добавляют своё. (Да, в более сложных концепциях можно безумного много наследовать AbstractEntity->StaticEntity->DynamicEntity->ActingEntity->CharacterEntity->SpaceMarine)

- `class SpaceMarine(Unit):` — в скобках указываем родителя
- `super().__init__(...)` — вызов `__init__` родителя
- Дочерний класс может **переопределить** метод родителя


### Пример


In [ ]:
class Unit:
    def __init__(self, name, wounds, armour_save):
        self.name = name
        self.wounds = wounds
        self.armour_save = armour_save

    def take_damage(self, damage):
        self.wounds -= damage
        if self.wounds <= 0:
            print(f'{self.name} уничтожен!')
        else:
            print(f'{self.name}: осталось {self.wounds} ран')

    def profile(self):
        print(f'{self.name} | Раны: {self.wounds} | Спасбросок: {self.armour_save}+')


class SpaceMarine(Unit):
    def __init__(self, name, chapter):
        super().__init__(name, wounds=2, armour_save=3)
        self.chapter = chapter

    def special_ability(self):
        print(f'{self.name}: And They Shall Know No Fear!')


class Tyranid(Unit):
    def __init__(self, name, synapse):
        super().__init__(name, wounds=1, armour_save=6)
        self.synapse = synapse

    def take_damage(self, damage):  # переопределяем метод родителя
        if self.synapse:
            print(f'Синапс держит {self.name} в строю!')
            damage = max(0, damage - 1)
        super().take_damage(damage)


titus = SpaceMarine('Brother Titus', 'Ultramarines')
gant = Tyranid('Termagant', synapse=True)

titus.profile()
gant.profile()
titus.special_ability()
gant.take_damage(2)
print(titus.__dict__)


In [ ]:
print(SpaceMarine.__mro__)
print(Tyranid.__mro__)


### 📝 Задание 3


In [ ]:
class Vehicle:
    def __init__(self, name, wounds, movement):
        self.name = name
        self.wounds = wounds
        self.movement = movement

    def move(self):
        print(f'{self.name} движется на {self.movement}"')


# 1. Создай класс Rhino(Vehicle)
#    Поля: capacity (вмещает бойцов, обычно 10)
#    super().__init__(name, wounds=10, movement=12)
#    Метод embark(unit_name) — 'Brother Titus садится в Rhino'
#
# 2. Создай класс LandRaider(Vehicle)
#    super().__init__(name, wounds=16, movement=10)
#    Переопредели move() — 'Land Raider Prometheus грохочет на 10"'
#
# 3. Создай объекты, проверь методы

# твой код здесь


In [ ]:
# Бонус:
# print(Rhino.__mro__)


### 🔍 Примечание

**Множественное наследование — когда всё идёт не так**

Python позволяет наследоваться от нескольких классов сразу:
```python
class TerminatorMarine(SpaceMarine, HeavyUnit):
    pass
```

Пока иерархия плоская — работает нормально. Но стоит добавить глубину и взаимозависимость — начинается настоящий Варп:

```python
class Unit: ...
class Marine(Unit): ...
class Psyker(Unit): ...
class LibrarianMarine(Marine, Psyker):  # оба родителя наследуют Unit
    ...
```

Теперь `Unit.__init__` вызывается дважды? Или один раз? Чей `take_damage` используется? Python решает это через **MRO (Method Resolution Order)** — алгоритм C3 linearization. Он строит чёткую очередь поиска, но читать её глазами при 4+ уровнях становится больно:

```python
print(LibrarianMarine.__mro__)
# [LibrarianMarine, Marine, Psyker, Unit, object]
# Python идёт слева направо — нашёл метод, остановился
```

**Что реально ломается:** если `Marine` и `Psyker` оба переопределяют `take_damage` по-разному, и ты вызываешь `super()` в цепочке — порядок вызовов становится неочевидным. Добавь ещё один уровень — и дебажить это удовольствие сомнительное.

**Практическое правило:** множественное наследование нормально для **миксинов** (небольших классов с одной ответственностью, без своего `__init__`). Глубокие взаимозависимые иерархии — признак что надо переосмыслить архитектуру. В реальных проектах предпочитают **композицию** вместо наследования:

```python
# вместо LibrarianMarine(Marine, Psyker)
class LibrarianMarine:
    def __init__(self, name):
        self.marine = Marine(name)   # содержит, а не является
        self.psyker = Psyker(name)
```


---
## Итог


| Концепция | Коротко |
|---|---|
| `__init__` | Ритуал инициации. Вызывается автоматически. Здесь задаём все поля. |
| `self` | Конвенция, не ключевое слово. Ссылка на конкретный объект. |
| Поля | Данные объекта. Живут в `self`, доступны из всех методов. |
| Методы | Действия объекта. `@classmethod`, `@staticmethod`, приватные `_`, магические `__dunder__` |
| Наследование | Дочерний класс получает всё от родителя. При множественном — следи за MRO. |
| Композиция | Альтернатива глубокому наследованию — объект *содержит* другие объекты. |


*Ave Imperator.*
